# 📅 2026-05-20 (수) 개발 노트 : 시맨틱 검색 + 점수 시스템 재설계 + 취향 분석 전면 개편

## 🎯 오늘의 목표
- [x] 자연어 시맨틱 검색 구현 (GPT 번역 + 임베딩)
- [x] 점수 시스템 재설계 (gem_percentile 백분위 정규화)
- [x] 매치율 상대 정규화 (1위=100%)
- [x] 레이더 차트 스케일 수정 (0~10)
- [x] 랭킹 페이지 gem 기준 정렬
- [x] 취향 분석 49개 전체 지표 + 카테고리 + 툴팁

## 🛠 진행 상황 및 핵심 기록

### 시맨틱 검색 v2 구현
- `POST /api/v1/games/search/semantic` 엔드포인트 추가
- 흐름: 한국어 쿼리 → GPT-4.1-mini 번역/지표 추출 → 영어 임베딩 → pgvector 검색
- `analyze_query()`: GPT로 영어 번역 + metric_hints 추출
  - 예: '혼자 조용히 즐기는 힐링' → `{english_query: 'cozy relaxing solo', metric_hints: {cozy_factor: 9}}`
- `embed_query()`: text-embedding-3-small로 1536차원 벡터 생성
- FINAL SCORE = 임베딩 유사도(60%) + gem_percentile(40%)
- 지표 힌트 10% 추가 반영
- 결과 상대 정규화: 1위=1.0, 최하위=0.5

### 점수 시스템 재설계
- **문제**: gem_potential이 70~80점대에 몰려 변별력 없음 (avg 75.6, p50=78)
- **해결**: `gem_percentile` 컬럼 추가 (백분위 기반 재분배)
  ```sql
  ALTER TABLE game_metrics ADD COLUMN gem_percentile FLOAT;
  UPDATE game_metrics SET gem_percentile = sub.percentile
  FROM (
    SELECT game_id, ROUND(PERCENT_RANK() OVER (ORDER BY gem_potential) * 100) as percentile
    FROM game_metrics WHERE gem_potential IS NOT NULL
  ) sub WHERE game_metrics.game_id = sub.game_id;
  ```
- **결과**: avg 43.4, p25=17, p50=48, p75=51 → 고르게 분포
- `models/game.py`에 `gem_percentile = Column(Float, nullable=True)` 추가
- `_calculate_gem_bonus()`: gem_percentile 우선 사용, 없으면 gem_potential 폴백
- 포맷 함수들: `gem_potential` 필드에 gem_percentile 값 반환

### 3계층 점수 설계
```
1. GEM SCORE (절대 품질) = gem_percentile → 랭킹, 뱃지
2. MATCH SCORE (취향 일치도) = similarity_score → 추천 결과
3. FINAL SCORE = GEM × 0.4 + MATCH × 0.6 → 시맨틱 검색 정렬
```

### 프론트엔드 수정
- `RadarChart.tsx`: `raw / 100` → `raw / 10` (0~10 스케일 정규화)
- `GameDetail.tsx`: `toPercent(norm)` → `value.toFixed(0) / 10` (10점 만점 표시)
- `RankingList.tsx`: gem_potential 기준 내림차순 정렬 추가
- `src/app/page.tsx`: `useSearch` → `semanticSearchGames` + submit 트리거 방식으로 변경
- `src/lib/api.ts`: `semanticSearchGames()` 함수 추가
- `src/lib/constants.ts`: `METRIC_DESCRIPTIONS` (49개 설명), `METRIC_CATEGORIES_KO` 추가
- `src/app/search/page.tsx`: 전면 개편
  - 8개 고정 슬라이더 → 49개 전체 지표 카테고리별 표시
  - ? 버튼 클릭 시 지표 설명 툴팁
  - 조정된 지표 요약 칩 표시
  - 중립값(5.0)과 다른 지표만 API 전송

### config.py 수정
- `OPENAI_API_KEY: str = ""` 추가 (.env에서 자동 주입)

### 주요 API 엔드포인트
```
POST /api/v1/games/search/semantic
     body: {query: string, limit: int, min_gem_potential: float}
     → RecommendationResponse (시맨틱 검색 결과)
```

## 🚨 트러블슈팅

### 1. 검색 결과 품질 낮음 (힐링 검색에 공포게임)
- **문제**: 한국어 임베딩 품질 낮음, 임베딩 모델이 영어 기반
- **원인**: '혼자 조용히' → 'silence' 연관으로 공포게임 매칭
- **해결**: GPT-4.1-mini로 영어 번역 후 임베딩 생성

### 2. 매치율 변별력 없음 (38%, 27%)
- **원인**: 코사인 유사도 절대값이 낮게 나옴
- **해결**: 상대 정규화 (최고점=1.0, 최저점=0.5로 스케일)
```python
def _normalize_scores(self, scored, score_idx=2):
    max_score = max(item[score_idx] for item in scored)
    min_score = min(item[score_idx] for item in scored)
    score_range = max_score - min_score
    norm = (raw_score - min_score) / score_range
    scaled = 0.5 + norm * 0.5  # 0.5~1.0 범위
```

### 3. 레이더 차트 너무 작게 표시
- **원인**: `value / 100` 으로 정규화 (0~10 스케일을 /100 하면 0.08 수준)
- **해결**: `value / 10` 으로 수정

### 4. 랭킹 점수 순서 안 맞음
- **원인**: similarity_score 순으로 정렬됨
- **해결**: `RankingList.tsx`에서 gem_potential 기준 내림차순 정렬
```typescript
const sorted = [...items].sort((a, b) => (b.gem_potential ?? 0) - (a.gem_potential ?? 0));
```

### 5. 랭킹 400 에러 (has_permadeath)
- **원인**: GENRE_PREFS에 Boolean 태그 `has_permadeath: 1` 포함
- **해결**: preferences는 수치 지표만, Boolean 태그 제거

## 💡 인사이트 및 다음 할 일

### 배운 점
- 임베딩 모델은 영어 기반 → 한국어 쿼리는 반드시 번역 후 임베딩
- 절대 유사도 점수보다 상대 정규화가 UX에 훨씬 효과적
- gem_potential 원본값보다 백분위가 변별력 있음
- FastAPI preferences 필드는 NUMERIC_METRIC_FIELDS만 허용 (Boolean 태그 불가)
- 3계층 점수 설계 (GEM/MATCH/FINAL)로 각 UI 맥락에 맞는 점수 사용

### 다음 할 일
- [ ] 메인 페이지 초기 AI 추천 게임 카드 정상 노출 확인
- [ ] 게임 상세 → 비슷한 게임 추천 확인
- [ ] 다크모드 완전 점검
- [ ] 모바일 반응형 점검
- [ ] Steam 로그인 구현
- [ ] '취향 DNA' 공유 카드 구현
- [ ] Vercel + Railway 배포
- [ ] 신작 게임 수집 파이프라인 (few-shot GPT-4.1-mini)